# Transformation

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_person AS 
SELECT 
  COALESCE(gender_concept.omop_concept_id, 0) AS gender_concept_id,
  YEAR(dbo_person.dateofbirth) AS year_of_birth,
  MONTH(dbo_person.dateofbirth) AS month_of_birth,
  DAY(dbo_person.dateofbirth) AS day_of_birth,
  dbo_person.dateofbirth AS birth_datetime,
  COALESCE(race_concept.omop_concept_id, 0) AS race_concept_id,
  COALESCE(ethnicity_concept.omop_concept_id, 0) AS ethnicity_concept_id,
  source_to_location.location_id AS location_id,
  NULL AS provider_id,
  NULL care_site_id,
  CONCAT_WS(CHR(31), 'allscripts_tw','dbo_person', 'id', dbo_person.id) AS person_source_value,
  dbo_sex_de.entryname AS gender_source_value,
  0 AS gender_source_concept_id,
  dbo_race_de.entryname AS race_source_value,
  0 AS race_source_concept_id,
  dbo_ethnicity_de.entryname AS ethnicity_source_value,
  0 AS ethnicity_source_concept_id,
  'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
ON dbo_person_other.id = dbo_person.id
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de
ON dbo_sex_de.id = dbo_person.sexde
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
ON dbo_race_de.id = dbo_person_other.racede
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
ON dbo_ethnicity_de.id = dbo_person_other.ethnicityde
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept gender_concept
ON gender_concept.source_id = dbo_person.sexde
AND gender_concept.domain_id = 'Gender'
AND gender_concept.source_system = 'allscripts_tw'
JOIN _exponent.omop_mapping.domain_source_to_concept race_concept
ON race_concept.source_id = dbo_person_other.racede
AND race_concept.domain_id = 'Race'
AND race_concept.source_system = 'allscripts_tw'
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept ethnicity_concept
ON ethnicity_concept.source_id = dbo_person_other.ethnicityde
AND ethnicity_concept.domain_id = 'Ethnicity'
AND ethnicity_concept.source_system = 'allscripts_tw'
LEFT OUTER JOIN _exponent.omop_mapping.source_to_location
ON source_to_location.location_source_value = CONCAT_WS(CHR(31), 'allscripts_tw','dbo_person', 'id', dbo_person.id)
WHERE 1=1
AND dbo_person.id IS NOT NULL
AND dbo_person.isinactiveflag = 'N'
AND dbo_person.dateofbirth IS NOT NULL
-- AND dbo_person.etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 14 DAY AND CURRENT_DATE()


In [0]:
%sql
MERGE INTO _exponent.omop_silver.person AS target
USING silver_person AS source
ON target.person_source_value = source.person_source_value

WHEN MATCHED AND (
     NOT target.gender_concept_id <=> source.gender_concept_id)
  OR NOT (target.year_of_birth <=> source.year_of_birth)
  OR NOT (target.month_of_birth <=> source.month_of_birth)
  OR NOT (target.day_of_birth <=> source.day_of_birth)
  OR NOT (target.birth_datetime <=> source.birth_datetime)
  OR NOT (target.race_concept_id <=> source.race_concept_id)
  OR NOT (target.ethnicity_concept_id <=> source.ethnicity_concept_id)
  OR NOT (target.location_id <=> source.location_id)
  OR NOT (target.provider_id <=> source.provider_id)
  OR NOT (target.care_site_id <=> source.care_site_id)
  OR NOT (target.gender_source_value <=> source.gender_source_value)
  OR NOT (target.gender_source_concept_id <=> source.gender_source_concept_id)
  OR NOT (target.race_source_value <=> source.race_source_value)
  OR NOT (target.race_source_concept_id <=> source.race_source_concept_id)
  OR NOT (target.ethnicity_source_value <=> source.ethnicity_source_value)
  OR NOT (target.ethnicity_source_concept_id <=> source.ethnicity_source_concept_id)
  OR NOT (target.source_system <=> source.source_system)

THEN UPDATE SET
 target.gender_concept_id           = source.gender_concept_id,
 target.year_of_birth               = source.year_of_birth,
 target.month_of_birth              = source.month_of_birth,
 target.day_of_birth                = source.day_of_birth,
 target.birth_datetime              = source.birth_datetime,
 target.race_concept_id             = source.race_concept_id,
 target.ethnicity_concept_id        = source.ethnicity_concept_id,
 target.location_id                 = source.location_id,
 target.provider_id                 = source.provider_id,
 target.care_site_id                = source.care_site_id,
 target.gender_source_value         = source.gender_source_value,
 target.gender_source_concept_id    = source.gender_source_concept_id,
 target.race_source_value           = source.race_source_value,
 target.race_source_concept_id      = source.race_source_concept_id,
 target.ethnicity_source_value      = source.ethnicity_source_value,
 target.ethnicity_source_concept_id = source.ethnicity_source_concept_id,
 target.source_system               = source.source_system,
 target.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  source.gender_concept_id,
  source.year_of_birth,
  source.month_of_birth,
  source.day_of_birth,
  source.birth_datetime,
  source.race_concept_id,
  source.ethnicity_concept_id,
  source.location_id,
  source.provider_id,
  source.care_site_id,
  source.person_source_value,
  source.gender_source_value,
  source.gender_source_concept_id,
  source.race_source_value,
  source.race_source_concept_id,
  source.ethnicity_source_value,
  source.ethnicity_source_concept_id,
  source.source_system,
  current_timestamp()
);


In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    person.source_system,
    person.person_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(person.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT source_system, person_source_value, last_mod_tsp
    FROM _exponent.omop_silver.person
) person
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person
  ON person.person_source_value = source_to_person.person_source_value;

In [0]:
%sql
-- MERGE INTO _exponent.omop.person AS gold_person
MERGE INTO _exponent.omop_tw.person AS gold_person
USING (
  SELECT
    source_to_person.person_id,                    
    person.gender_concept_id,
    person.year_of_birth,
    person.month_of_birth,
    person.day_of_birth,
    person.birth_datetime,
    person.race_concept_id,
    person.ethnicity_concept_id,
    person.location_id,
    person.provider_id,
    person.care_site_id,
    person.person_source_value,
    person.gender_source_value,
    person.gender_source_concept_id,
    person.race_source_value,
    person.race_source_concept_id,
    person.ethnicity_source_value,
    person.ethnicity_source_concept_id
  FROM _exponent.omop_silver.person
  JOIN _exponent.omop_mapping.source_to_person
    ON source_to_person.person_source_value = person.person_source_value
   AND source_to_person.active_flag = TRUE
  WHERE 1=1
  AND person.source_system = 'allscripts_tw'
) AS src
ON gold_person.person_id = src.person_id

WHEN MATCHED AND (
     NOT (gold_person.gender_concept_id <=> src.gender_concept_id)
  OR NOT (gold_person.year_of_birth <=> src.year_of_birth)
  OR NOT (gold_person.month_of_birth <=> src.month_of_birth)
  OR NOT (gold_person.day_of_birth <=> src.day_of_birth)
  OR NOT (gold_person.birth_datetime <=> src.birth_datetime)
  OR NOT (gold_person.race_concept_id <=> src.race_concept_id)
  OR NOT (gold_person.ethnicity_concept_id <=> src.ethnicity_concept_id)
  OR NOT (gold_person.location_id <=> src.location_id)
  OR NOT (gold_person.provider_id <=> src.provider_id)
  OR NOT (gold_person.care_site_id <=> src.care_site_id)
  OR NOT (gold_person.gender_source_value <=> src.gender_source_value)
  OR NOT (gold_person.gender_source_concept_id <=> src.gender_source_concept_id)
  OR NOT (gold_person.race_source_value <=> src.race_source_value)
  OR NOT (gold_person.race_source_concept_id <=> src.race_source_concept_id)
  OR NOT (gold_person.ethnicity_source_value <=> src.ethnicity_source_value)
  OR NOT (gold_person.ethnicity_source_concept_id <=> src.ethnicity_source_concept_id)

) THEN UPDATE SET
  gold_person.gender_concept_id = src.gender_concept_id,
  gold_person.year_of_birth = src.year_of_birth,
  gold_person.month_of_birth = src.month_of_birth,
  gold_person.day_of_birth = src.day_of_birth,
  gold_person.birth_datetime = src.birth_datetime,
  gold_person.race_concept_id = src.race_concept_id,
  gold_person.ethnicity_concept_id = src.ethnicity_concept_id,
  gold_person.location_id = src.location_id,
  gold_person.provider_id = src.provider_id,
  gold_person.care_site_id = src.care_site_id,
  gold_person.person_source_value = src.person_source_value,
  gold_person.gender_source_value = src.gender_source_value,
  gold_person.gender_source_concept_id = src.gender_source_concept_id,
  gold_person.race_source_value = src.race_source_value,
  gold_person.race_source_concept_id = src.race_source_concept_id,
  gold_person.ethnicity_source_value = src.ethnicity_source_value,
  gold_person.ethnicity_source_concept_id = src.ethnicity_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id
)
VALUES (
  src.person_id,
  src.gender_concept_id,
  src.year_of_birth,
  src.month_of_birth,
  src.day_of_birth,
  src.birth_datetime,
  src.race_concept_id,
  src.ethnicity_concept_id,
  src.location_id,
  src.provider_id,
  src.care_site_id,
  src.person_source_value,
  src.gender_source_value,
  src.gender_source_concept_id,
  src.race_source_value,
  src.race_source_concept_id,
  src.ethnicity_source_value,
  src.ethnicity_source_concept_id
);


In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_tw.person;
-- TRUNCATE TABLE _exponent.omop_silver.person;
-- TRUNCATE TABLE _exponent.omop_mapping.source_to_person;